# 06 · Explainability laboratory
**OMARI Seminar Series — From Pixels to Predictions: Developing AI Models for Medical Image Interpretation**

Compute CAM, Grad-CAM and Grad-CAM++ for DenseNet-161 and transformer-appropriate attributions (token Grad-CAM, attention rollout, gradient-weighted rollout) for both ViTs; plug in a MetaCAM++ implementation if available; test faithfulness (deletion/insertion, cascading randomisation, perturbation stability); export genuine heatmaps for the website.

| | |
|---|---|
| **Runtime** | GPU recommended (attributions need backward passes); demo ≈10 min |
| **Needs** | fine-tuned checkpoints from notebooks 02–04 |
| **Produces** | figures, `faithfulness.csv`, and (optionally) the full website bundle |
| **Modes** | `smoke` (synthetic data, random weights: checks the code only) · `demo` (real data subset, few epochs) · `full` (complete experiment) |

> Results from `smoke` mode are meaningless by construction and must never be exported to the website.
> Every artifact written by these notebooks records the script, checkpoint hash and parameters that produced it.

In [ ]:
#@title 0 · Setup: repository, packages, data and artifact folders { display-mode: "form" }
import os, sys, subprocess, json, random, time
MODE = "demo"  #@param ["smoke", "demo", "full"]
REPO_URL = "https://github.com/YOUR-ORG/omari-workshop.git"  #@param {type:"string"}
DATA_ROOT = "/content/data/FETAL_PLANES_ZENODO"  #@param {type:"string"}
ARTIFACTS = "/content/omari_artifacts"  #@param {type:"string"}
USE_DRIVE_FOR_ARTIFACTS = False  #@param {type:"boolean"}
MODE = os.environ.get("OMARI_MODE", MODE)            # lets CI / nbclient override
ARTIFACTS = os.environ.get("OMARI_ARTIFACTS", ARTIFACTS)
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and USE_DRIVE_FOR_ARTIFACTS:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACTS = "/content/drive/MyDrive/omari_artifacts"

# 1) make `omari_us` importable: installed package > local checkout (../python) > clone REPO_URL
def _find_repo():
    for cand in [os.getcwd(), os.path.dirname(os.getcwd()), "/content/omari-workshop"]:
        if os.path.isdir(os.path.join(cand, "python", "omari_us")):
            return cand
    return None
REPO = _find_repo()
if REPO is None:
    if "YOUR-ORG" in REPO_URL:
        raise RuntimeError("Set REPO_URL to the repository that contains python/omari_us (or upload the repo to /content/omari-workshop).")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/omari-workshop"], check=True)
    REPO = "/content/omari-workshop"
sys.path.insert(0, os.path.join(REPO, "python"))

# 2) packages (Colab already has torch/torchvision; timm and friends are added)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm>=0.9.12", "scikit-learn", "scipy", "pyyaml", "opencv-python-headless", "gdown"], check=True)

import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import omari_us
from omari_us.utils import seed_everything, get_device, CLASSES, CLASS_SHORT
SEED = 42
seed_everything(SEED)
DEVICE = get_device()
os.makedirs(ARTIFACTS, exist_ok=True)
SPLITS_CSV = os.path.join(ARTIFACTS, "splits", "split_manifest.csv")
RUNS = os.path.join(ARTIFACTS, "runs")
print(f"omari_us {omari_us.__version__} | torch {torch.__version__} | device {DEVICE} | MODE={MODE} | artifacts → {ARTIFACTS}")
if DEVICE.type == "cpu" and MODE != "smoke":
    print("⚠ No GPU. In Colab: Runtime → Change runtime type → T4 GPU. CPU training of these models is impractically slow.")

In [ ]:
#@title 1 · Data: download FETAL_PLANES_DB (2.1 GB, CC BY 4.0) or build the synthetic smoke dataset { display-mode: "form" }
from omari_us.datasets import read_metadata, make_splits, synthetic_metadata, save_split_manifest, load_split_manifest, ZENODO_URL, ZENODO_MD5

def ensure_fetal_planes(root=DATA_ROOT):
    """Download + verify + extract the Zenodo archive if the metadata CSV is not already present."""
    import glob, hashlib
    if glob.glob(os.path.join(root, "**", "FETAL_PLANES_DB_data.csv"), recursive=True):
        return
    os.makedirs(root, exist_ok=True)
    z = os.path.join(os.path.dirname(root), "FETAL_PLANES_ZENODO.zip")
    if not os.path.exists(z):
        print("downloading", ZENODO_URL)
        subprocess.run(["curl", "-L", "--fail", "-o", z, ZENODO_URL], check=True)
    md5 = hashlib.md5(open(z, "rb").read()).hexdigest()
    assert md5 == ZENODO_MD5, f"md5 mismatch: {md5}"
    subprocess.run(["unzip", "-q", "-n", z, "-d", root], check=True)

if MODE == "smoke":
    SMOKE_DIR = os.path.join(ARTIFACTS, "smoke_data")
    meta = synthetic_metadata(60, (2, 5), seed=0, image_dir=os.path.join(SMOKE_DIR, "Images"), size=(200, 150))
    print("SMOKE MODE: synthetic grey images; every number below is meaningless.")
else:
    ensure_fetal_planes()
    meta = read_metadata(DATA_ROOT)
print(meta.shape)
meta.head()

In [ ]:
# Load the split manifest written by notebook 01 (or rebuild it identically from the metadata)
if os.path.exists(SPLITS_CSV) and MODE != "smoke":
    splits = load_split_manifest(SPLITS_CSV, DATA_ROOT)
else:
    splits = make_splits(meta, val_fraction=0.15 if MODE == "smoke" else 0.1, seed=SEED)
    save_split_manifest(splits, SPLITS_CSV)
if MODE == "demo":  # stratified patient subset for speed
    keep = splits.groupby("split", group_keys=False).apply(lambda d: d[d.Patient_num.isin(pd.Series(d.Patient_num.unique()).sample(frac=0.25, random_state=SEED))])
    splits = keep.reset_index(drop=True)
print(splits.groupby(["split"]).agg(images=("Image_name", "size"), patients=("Patient_num", "nunique")))

In [ ]:
import glob
from omari_us.models import load_finetuned
from omari_us.datasets import input_spec, build_transforms, denormalize
from omari_us import explainability as X
from PIL import Image
MODELS = {}
for mid in ["densenet161", "vit_b16", "usf_mae"]:
    c = sorted(glob.glob(os.path.join(RUNS, f"{mid}_finetune_lf1_s0*", "best.pt")))
    if c:
        m, meta_ = load_finetuned(mid, c[0]); MODELS[mid] = (m.to(DEVICE).eval(), build_transforms(input_spec(mid), False))
print("loaded:", list(MODELS))
print("method compatibility:", {m: X.methods_for(m) for m in MODELS})

## 1 · Methods and their assumptions
| Method | Needs | Class-specific | Here |
|---|---|---|---|
| CAM | conv features → GAP → linear | yes | DenseNet-161 (`model.act`, 2208×7×7) |
| Grad-CAM | any (B,C,H,W) activation | yes | DenseNet any block; ViT via token reshape |
| Grad-CAM++ | as Grad-CAM; exp-score closed form | yes | same |
| Attention rollout | attention matrices | **no** | ViT-B/16, USF-MAE |
| Gradient-weighted rollout | attention + gradients | yes | ViT-B/16, USF-MAE |
| MetaCAM++ | — | — | reserved; supply an implementation below |

For a GAP-linear head, Grad-CAM = ReLU(CAM)/Z exactly (verified in `tests/test_models_and_xai.py`).

In [ ]:
row = splits[splits.split == "test"].iloc[0]
img = Image.open(row.path).convert("L")
target = int(row.label)
cols = [(m, meth) for m in MODELS for meth in X.methods_for(m)]
fig, axes = plt.subplots(1, len(cols) + 1, figsize=(2.6 * (len(cols) + 1), 3))
axes[0].imshow(img.resize((224, 224)), cmap="gray"); axes[0].set_title(f"true: {CLASS_SHORT[target]}", fontsize=8)
for ax, (m, meth) in zip(axes[1:], cols):
    model, tf = MODELS[m]; x = tf(img).unsqueeze(0).to(DEVICE)
    a = X.normalise(X.upsample(X.explain(meth, m, model, x, target)))
    ax.imshow(img.resize((224, 224)), cmap="gray"); ax.imshow(a, cmap="inferno", alpha=.55); ax.set_title(f"{m}\n{meth}", fontsize=7)
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## 2 · MetaCAM++ (user-supplied)
This cell registers your implementation. Nothing is implemented here; leave `METACAMPP_PATH` empty to skip.

In [ ]:
METACAMPP_PATH = ""  #@param {type:"string"}
if METACAMPP_PATH:
    import importlib.util
    spec_ = importlib.util.spec_from_file_location("metacampp", METACAMPP_PATH); mod = importlib.util.module_from_spec(spec_); spec_.loader.exec_module(mod)
    # the module is expected to call omari_us.explainability.register_method("metacampp", models=(...))
    print("registered:", X.compatible("metacampp", "densenet161"), X.compatible("metacampp", "vit_b16"))
else:
    print("MetaCAM++ not supplied — reserved slot only.")

## 3 · Faithfulness: deletion and insertion
Remove (or reveal) pixels in attribution order, replacing them with a heavy blur. Compare with a random order.

In [ ]:
N_IMAGES = 30 if MODE == "full" else (2 if MODE == "smoke" else 8)
test_rows = splits[splits.split == "test"].sample(N_IMAGES, random_state=SEED)
recs = []
for _, r in test_rows.iterrows():
    im = Image.open(r.path).convert("L")
    for m, (model, tf) in MODELS.items():
        x = tf(im).unsqueeze(0).to(DEVICE)
        for meth in X.methods_for(m):
            a = X.explain(meth, m, model, x, int(r.label))
            d = X.faithfulness.deletion_insertion(model, x, a, int(r.label), steps=10, mode="deletion")
            ins = X.faithfulness.deletion_insertion(model, x, a, int(r.label), steps=10, mode="insertion")
            recs.append({"model": m, "method": meth, "deletion_auc": d["auc"], "insertion_auc": ins["auc"]})
        rnd = torch.rand(14, 14)
        recs.append({"model": m, "method": "random", "deletion_auc": X.faithfulness.deletion_insertion(model, x, rnd, int(r.label), steps=10)["auc"],
                     "insertion_auc": X.faithfulness.deletion_insertion(model, x, rnd, int(r.label), steps=10, mode="insertion")["auc"]})
fa = pd.DataFrame(recs).groupby(["model", "method"]).agg(["mean", "std"]).round(3)
fa.to_csv(os.path.join(ARTIFACTS, "faithfulness.csv")); fa

**Read:** lower deletion AUC and higher insertion AUC than `random` indicate the ranking carries information about *this model's* decision. They say nothing about clinical correctness.

## 4 · Sanity check: cascading randomisation (Adebayo et al., 2018)

In [ ]:
if "densenet161" in MODELS:
    model, tf = MODELS["densenet161"]; x = tf(img).unsqueeze(0).to(DEVICE)
    groups = ["classifier", "features.norm5", "features.denseblock4", "features.denseblock3", "features.denseblock2"]
    res = X.faithfulness.cascading_randomisation(model, x, target, lambda mm: X.gradcam(mm, x, target, mm.act), groups)
    display(pd.DataFrame(res))
for m in [k for k in MODELS if k != "densenet161"]:
    model, tf = MODELS[m]; x = tf(img).unsqueeze(0).to(DEVICE)
    groups = ["head", "norm", "blocks.11", "blocks.10", "blocks.9", "blocks.6"]
    res = X.faithfulness.cascading_randomisation(model, x, target, lambda mm: X.vitx.token_gradcam(mm, x, target), groups)
    print(m); display(pd.DataFrame(res))

## 5 · Perturbation stability
Same image under the perturbation grid used by the website; rank correlation with the unperturbed map and change in the true-class probability.

In [ ]:
from omari_us.datasets import perturbations as PT
g = np.asarray(img, dtype=np.float32) / 255.0
rows = []
for kind, param, label in PT.DEFAULT_GRID:
    pg = PT.apply(g, kind, param, seed=0)
    pim = Image.fromarray((pg * 255).astype(np.uint8))
    for m, (model, tf) in MODELS.items():
        meth = "gradcam" if m == "densenet161" else "vit_gradcam"
        x0, x1 = tf(img).unsqueeze(0).to(DEVICE), tf(pim).unsqueeze(0).to(DEVICE)
        with torch.no_grad(): p0, p1 = [torch.softmax(model(x), -1)[0, target].item() for x in (x0, x1)]
        rho = X.faithfulness.spearman(X.explain(meth, m, model, x0, target), X.explain(meth, m, model, x1, target))
        rows.append({"perturbation": label, "model": m, "Δp(true)": p1 - p0, "map ρ": rho})
pd.DataFrame(rows).pivot(index="perturbation", columns="model").round(3)

## 6 · Export the full website bundle
Writes predictions, feature maps, attention, attributions, reconstructions and perturbation experiments with provenance (see `configs/export.yaml`).

In [ ]:
EXPORT = False  #@param {type:"boolean"}
if EXPORT and MODE != "smoke":
    from omari_us.export import Exporter
    from omari_us.utils import load_yaml
    cfg = load_yaml(os.path.join(REPO, "configs", "export.yaml"))
    cfg["output_dir"] = os.path.join(ARTIFACTS, "website_bundle")
    cfg["models"] = {m: {"checkpoint": sorted(glob.glob(os.path.join(RUNS, f"{m}_finetune_lf1_s0*", "best.pt")))[0]} for m in MODELS}
    ck = os.path.join(ARTIFACTS, "checkpoints", "usf_mae_100ep.pth")
    if "usf_mae" in cfg["models"] and os.path.exists(ck): cfg["models"]["usf_mae"]["pretrain_checkpoint"] = ck
    cfg["evaluation"]["runs"] = {m: os.path.dirname(v["checkpoint"]) for m, v in cfg["models"].items()}
    cfg["evaluation"].pop("curve_runs", None); cfg["evaluation"]["label_efficiency_glob"] = os.path.join(RUNS, "*", "summary.json")
    Exporter(cfg).run(splits, clean=True)
    print("Zip", cfg["output_dir"], "and unpack it into website/public/experiments/")

## References
- Burgos-Artizzu X.P. et al. *Evaluation of deep convolutional neural networks for automatic classification of common maternal fetal ultrasound planes.* Sci. Rep. 10, 10200 (2020). Dataset: Zenodo 3904280, CC BY 4.0.
- Huang G. et al. *Densely connected convolutional networks.* CVPR 2017.
- Dosovitskiy A. et al. *An image is worth 16×16 words.* ICLR 2021; Steiner A. et al. *How to train your ViT?* TMLR 2022.
- He K. et al. *Masked autoencoders are scalable vision learners.* CVPR 2022.
- Megahed Y. et al. *USF-MAE: Ultrasound self-supervised foundation model with masked autoencoding.* Biomed. Signal Process. Control 122, 110313 (2026). https://github.com/Yusufii9/USF-MAE
- Zhou B. et al. CAM, CVPR 2016 · Selvaraju R.R. et al. Grad-CAM, ICCV 2017 · Chattopadhay A. et al. Grad-CAM++, WACV 2018 · Abnar & Zuidema, attention rollout, ACL 2020 · Chefer H. et al., CVPR 2021 · Adebayo J. et al., sanity checks, NeurIPS 2018 · Petsiuk V. et al., RISE / deletion–insertion, BMVC 2018 · Guo C. et al., calibration, ICML 2017.